# IndexTTS-2.5 · Google Colab Pro L4

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/IndexTTS2_5_ColabPro_L4_UV_20260817.ipynb)

固定官方 `v2.5.0`，使用 `uv` 锁定依赖；支持 BF16、多语言音色克隆、情感向量、语速控制，并启动带 `gradio.live` 临时链接的 Gradio WebUI。

> 先在「运行时 → 更改运行时类型」选择 GPU/L4。只使用本人或已获明确授权的声音，并阅读上游许可与免责声明。


In [ ]:
import os, shutil, subprocess, sys, urllib.request
from pathlib import Path

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, env=os.environ.copy(), check=True)

smi=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv,noheader"],text=True,capture_output=True)
if smi.returncode: raise RuntimeError("未检测到 NVIDIA GPU，请切换 Colab GPU 运行时。")
print(smi.stdout.strip())
if "L4" not in smi.stdout: print("⚠️ 当前不是 L4，仍可尝试运行。")

TAG="v2.5.0" #@param {type:"string"}
REPO=Path("/content/index-tts")
USE_DRIVE=False #@param {type:"boolean"}
HF_ENDPOINT="" #@param {type:"string"}
if HF_ENDPOINT.strip(): os.environ["HF_ENDPOINT"]=HF_ENDPOINT.strip().rstrip("/")
os.environ["UV_LINK_MODE"]="copy"


## 安装

首次安装和下载模型需数分钟。仅安装 `webui` 额外依赖，不安装易在 Colab 编译失败的 DeepSpeed/FlashAttention。


In [ ]:
run(["apt-get","update","-qq"])
run(["apt-get","install","-y","-qq","git","ffmpeg"])
if (REPO/".git").exists():
    run(["git","fetch","--tags","--force","--depth","1","origin"],REPO)
    run(["git","checkout","--force",TAG],REPO)
else:
    if REPO.exists(): shutil.rmtree(REPO)
    run(["git","clone","--depth","1","--branch",TAG,"https://github.com/index-tts/index-tts.git",REPO])
run([sys.executable,"-m","pip","install","-q","-U","uv"])
run(["uv","sync","--extra","webui","--frozen"],REPO)


## 下载 IndexTTS-2.5 模型

可选 Google Drive 缓存会把模型保存到 `MyDrive/AI-Models/IndexTTS-2.5`。


In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    MODEL_DIR=Path("/content/drive/MyDrive/AI-Models/IndexTTS-2.5")
else:
    MODEL_DIR=REPO/"checkpoints"
MODEL_DIR.mkdir(parents=True,exist_ok=True)
program=f'''from indextts.utils.model_download import snapshot_download,ensure_config_available,ensure_models_available
m={str(MODEL_DIR)!r}
snapshot_download("IndexTeam/IndexTTS-2.5",local_dir=m)
ensure_config_available(m,version="2.5")
ensure_models_available(m)'''
run(["uv","run","python","-c",program],REPO)
required=["config.yaml","gpt.pth","s2mel.pth","codec.pth","multilingual_zh_ja_yue_char_del.tiktoken","wav2vec2bert_stats.pt"]
missing=[x for x in required if not (MODEL_DIR/x).is_file()]
if missing: raise FileNotFoundError("模型不完整："+", ".join(missing))
run(["uv","run","python","tools/gpu_check.py"],REPO)
print("✅ 模型就绪：",MODEL_DIR)


## 启动 WebUI

直接运行本单元格；参考音频在 WebUI 中上传，无需使用 Colab 的 `files.upload()`。等待输出 `https://……gradio.live` 临时链接后打开。此单元格会持续运行，用完点击停止。L4 WebUI 会加载文本情感模型，显存占用较高。


In [ ]:
launcher=REPO/"colab_webui_v25.py"
urllib.request.urlretrieve("https://raw.githubusercontent.com/zencolab/WhatDreamsCost-ComfyUI/main/indextts25_colabpro_l4_webui_20260817.py",launcher)
run(["uv","run","python",launcher,"--model_dir",MODEL_DIR,"--port",7860],REPO)


## 发音与故障排查

- 中文：`他在银<行|XING2>里<行|HANG2>走。`
- 英文：`a <minute|M IH1 . N AH0 T>`
- 日语：`<上手|じょうず>`
- 下载中断：重跑模型单元格；网络受限可设置 `HF_ENDPOINT=https://hf-mirror.com`。
- OOM：停止 WebUI、重启运行时，缩短文本或参考音频后重试。

来源：[官方仓库](https://github.com/index-tts/index-tts) · [中文文档](https://github.com/index-tts/index-tts/blob/main/docs/README_zh.md) · [v2.5.0](https://github.com/index-tts/index-tts/releases/tag/v2.5.0)。本 Notebook 不分发模型权重。
